# 01 — METR-LA Gerçek Veri Seti: Keşifsel Veri Analizi (EDA) ve Veri Temizleme Raporu

**Tez Konusu:** Zaman Serisi Analizi ve Derin Öğrenme Yöntemleriyle Dinamik Trafik Akışı Tahmini  
**Veri Kaynağı:** METR-LA (Los Angeles County Metropolitan Transportation Authority — 207 Otoyol Döngü Dedektörü, 5 Dakikalık Örnekleme)

Bu notebook, tez çalışmasının **Veri Seti ve Ön İşleme (Metodoloji)** bölümü için gerekli tüm istatistiksel doğrulamaları, eksik veri / aykırı değer analizlerini, uzamsal-zamansal bağımlılık kanıtlarını ve veri sızıntısız (zero-leakage) normalizasyon adımlarını içerir.

## 1. Veri Setinin Yüklenmesi ve Temel Tanımlayıcı İstatistikler

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('../src').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import (
    load_metr_la_dataframe,
    load_sensor_locations,
    build_spatial_weight_matrix,
    prepare_spatiotemporal_arrays,
)

sns.set_theme(style='whitegrid', context='notebook')

df = load_metr_la_dataframe()
locations = load_sensor_locations(list(df.columns))
adj = build_spatial_weight_matrix(list(df.columns))

total_cells = df.size
zero_cells = int((df.values == 0.0).sum())
nan_cells = int(df.isna().sum().sum())
nz = df.values[df.values > 0.0]

summary_table = pd.DataFrame({
    'Özellik': [
        'Zaman Adımı Sayısı (5 dk aralık)', 'Sensör Sayısı', 'Toplam Gözlem Hücresi',
        'Başlangıç Tarihi', 'Bitiş Tarihi', 'NaN Sayısı',
        'Eksik Sensör Okuması (0.0 mph)', 'Ortalama Hız (Sıfır Hariç, mph)',
        'Standart Sapma (mph)', 'Medyan / Q1 / Q3 (mph)', 'Min / Max Hız (mph)'
    ],
    'Değer': [
        f'{df.shape[0]:,}', f'{df.shape[1]}', f'{total_cells:,}',
        str(df.index.min()), str(df.index.max()), f'{nan_cells}',
        f'{zero_cells:,} (%{zero_cells / total_cells * 100:.2f})',
        f'{nz.mean():.2f}', f'{nz.std():.2f}',
        f'{np.median(nz):.2f} / {np.percentile(nz, 25):.2f} / {np.percentile(nz, 75):.2f}',
        f'{nz.min():.2f} / {nz.max():.2f}'
    ]
})
summary_table

## 2. Eksik Veri (`0.0`) ve Sensör Kopma Analizi
METR-LA literatüründe (Li vd., DCRNN 2018) `0.0 mph` değerleri gerçek tam durma değil, **sensör iletişim/donanım kopmalarını** temsil eder. Aşağıda sensör bazlı eksik veri oranları ve fiziksel tutarlılık (aykırı değer) kontrolleri yapılmaktadır.

In [ ]:
zero_rate_per_sensor = (df == 0.0).mean(axis=0) * 100.0
neg_count = int((df.values < 0.0).sum())
over_70_count = int((df.values > 70.0).sum())

print(f'Negatif hız sayısı          : {neg_count}')
print(f'70 mph üstü aykırı değer    : {over_70_count}')
print(f'Sensör başı eksik oran (Ort): %{zero_rate_per_sensor.mean():.2f} | Medyan: %{zero_rate_per_sensor.median():.2f} | Max: %{zero_rate_per_sensor.max():.2f}')
print('\nEn yüksek eksik okuma oranına sahip ilk 5 sensör:')
print(zero_rate_per_sensor.sort_values(ascending=False).head(5).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(zero_rate_per_sensor, bins=25, color='#ef4444', alpha=0.8, edgecolor='white')
axes[0].set_title('Sensör Bazlı Eksik Veri (0.0) Oranı Dağılımı')
axes[0].set_xlabel('Eksik Veri Oranı (%)')
axes[0].set_ylabel('Sensör Sayısı')

daily_missing = (df == 0.0).mean(axis=1).resample('D').mean() * 100.0
axes[1].plot(daily_missing.index, daily_missing.values, color='#f59e0b', lw=1.6)
axes[1].set_title('Günlük Ortalama Sensör Kopma Oranı (%)')
axes[1].set_ylabel('Eksik Oran (%)')
plt.tight_layout()
plt.show()

## 3. Veri Temizleme (Imputation) & Maskeleme Stratejisi: Öncesi vs. Sonrası
Bilimsel geçerlilik için ikili strateji uygulanır:
1. **Model Girdileri (`X`):** `0.0` değerleri `NaN` yapılır; kısa süreli kopmalar (≤ 30 dk / 6 adım) **zamansal lineer interpolasyon** (`interpolate(method='time', limit=6)`) ile fiziksel olarak yumuşak doldurulur, daha uzun kopmalarda `ffill/bfill` ve sensör ortalaması kullanılır.
2. **Model Hedefleri (`Y`) ve Değerlendirme Metrikleri:** Yapay doldurulmuş değerlerden haksız doğruluk çıkmaması için eğitim kaybı ve test metrikleri (**Masked MAE, RMSE, MAPE, R²**) yalnızca ham veride `> 0.0` olan gerçek ölçümler üzerinden hesaplanır.

In [ ]:
data = prepare_spatiotemporal_arrays(df, adj)
raw_nz = data['raw'][data['raw'] > 0.0]
filled_all = data['filled'].ravel()

clean_cmp = pd.DataFrame({
    'Metrik': ['Ortalama (mph)', 'Standart Sapma (mph)', 'Medyan (mph)', 'Q25 (mph)', 'Q75 (mph)', 'Eksik / Sıfır Oranı (%)'],
    'Ham Veri (Tüm Hücreler)': [
        round(float(data['raw'].mean()), 3), round(float(data['raw'].std()), 3),
        round(float(np.median(data['raw'])), 3), round(float(np.percentile(data['raw'], 25)), 3),
        round(float(np.percentile(data['raw'], 75)), 3), round(float((data['raw'] == 0).mean() * 100), 2)
    ],
    'Ham Veri (Sıfır Hariç Gerçek)': [
        round(float(raw_nz.mean()), 3), round(float(raw_nz.std()), 3),
        round(float(np.median(raw_nz)), 3), round(float(np.percentile(raw_nz, 25)), 3),
        round(float(np.percentile(raw_nz, 75)), 3), 0.0
    ],
    'Temizlenmiş Girdi (Interpolasyon + ffill)': [
        round(float(filled_all.mean()), 3), round(float(filled_all.std()), 3),
        round(float(np.median(filled_all)), 3), round(float(np.percentile(filled_all, 25)), 3),
        round(float(np.percentile(filled_all, 75)), 3), 0.0
    ]
})
clean_cmp

## 4. Zamansal (Temporal) Örüntü ve Otokorelasyon Analizi
Sabah/akşam zirve saatleri, hafta içi–hafta sonu ayrımı ve gecikme (lag) otokorelasyonu; **12 adımlık (60 dk) geçmiş pencere** ve **döngüsel zaman (sin/cos)** özelliklerinin gerekliliğini kanıtlar.

In [ ]:
df_nz = df.replace(0.0, np.nan)
mean_speed = df_nz.mean(axis=1)
eda = pd.DataFrame({
    'speed': mean_speed,
    'hour': df.index.hour + df.index.minute / 60.0,
    'dow': df.index.day_of_week,
    'is_weekend': df.index.day_of_week >= 5
})

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.5))
eda[~eda['is_weekend']].groupby('hour')['speed'].mean().plot(ax=axes[0], lw=2.2, color='#2563eb', label='Hafta İçi')
eda[eda['is_weekend']].groupby('hour')['speed'].mean().plot(ax=axes[0], lw=2.2, color='#f59e0b', linestyle='--', label='Hafta Sonu')
axes[0].axvspan(7, 9.5, color='#ef4444', alpha=0.08, label='Sabah Zirve')
axes[0].axvspan(16, 19, color='#dc2626', alpha=0.08, label='Akşam Zirve')
axes[0].set_title('Hafta İçi vs. Hafta Sonu Günlük Hız Profili')
axes[0].set_xlabel('Günün Saati')
axes[0].set_ylabel('Ortalama Hız (mph)')
axes[0].legend()

# Otokorelasyon (15 dk=3, 30 dk=6, 60 dk=12, 24 saat=288 lag)
series_clean = pd.Series(data['filled'].mean(axis=1))
lags = [1, 3, 6, 12, 24, 72, 144, 288]
acfs = [series_clean.autocorr(lag=l) for l in lags]
lag_labels = ['5dk', '15dk', '30dk', '60dk', '2sa', '6sa', '12sa', '24sa']
axes[1].bar(lag_labels, acfs, color='#7c3aed', alpha=0.85)
axes[1].set_title('Ağ Ortalaması Hız Otokorelasyonu (Lag ACF)')
axes[1].set_ylabel('Otokorelasyon (r)')
for i, v in enumerate(acfs):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Uzamsal (Spatial) Yol Ağı Bağımlılığı ve Kronolojik Bölme Doğrulaması
Yol ağı üzerinde komşu olan sensörlerin hız korelasyonu, rastgele uzak sensör çiftleriyle karşılaştırılarak **uzamsal komşuluk özelliğinin** bilimsel gerekçesi gösterilir. Son olarak %70 / %10 / %20 kronolojik bölme aralıkları doğrulanır.

In [ ]:
corr_mx = np.corrcoef(data['filled'].T)
nbr_mask = adj > 0
non_nbr_mask = (adj == 0) & (~np.eye(len(adj), dtype=bool))

print(f'Komşu Sensör Çiftleri Ortalama Hız Korelasyonu : {corr_mx[nbr_mask].mean():.4f}')
print(f'Komşu Olmayan Sensör Çiftleri Ort. Korelasyonu : {corr_mx[non_nbr_mask].mean():.4f}')
print(f'Sensör Başı Ortalama Komşu Sayısı              : {(adj > 0).sum(axis=1).mean():.2f}')

tr_end, va_end = data['train_end'], data['val_end']
ts = data['timestamps']
split_df = pd.DataFrame([
    {'Bölme': 'Eğitim (%70)', 'Başlangıç': str(ts[0]), 'Bitiş': str(ts[tr_end - 1]), 'Adım': tr_end,
     'Ort. Hız (Sıfır Hariç)': round(float(data['raw'][:tr_end][data['raw'][:tr_end] > 0].mean()), 2)},
    {'Bölme': 'Doğrulama (%10)', 'Başlangıç': str(ts[tr_end]), 'Bitiş': str(ts[va_end - 1]), 'Adım': va_end - tr_end,
     'Ort. Hız (Sıfır Hariç)': round(float(data['raw'][tr_end:va_end][data['raw'][tr_end:va_end] > 0].mean()), 2)},
    {'Bölme': 'Test (%20)', 'Başlangıç': str(ts[va_end]), 'Bitiş': str(ts[-1]), 'Adım': len(ts) - va_end,
     'Ort. Hız (Sıfır Hariç)': round(float(data['raw'][va_end:][data['raw'][va_end:] > 0].mean()), 2)},
])
print(f"\nZ-Score Normalizasyon (Sadece Eğitim Seti): mean={data['mean']:.4f} mph, std={data['std']:.4f} mph")
split_df